<a href="https://colab.research.google.com/github/ShymaShameer/Eniac_project/blob/main/Project_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project**
**Case Study: Eniac’s Discount Strategy**


# Data

In [1]:
import pandas as pd

In [2]:
pd.set_option('display.max_colwidth', None) # to increase the width of the columns

In [3]:
url = 'https://drive.google.com/file/d/1m1ThDDIYRTTii-rqM5SEQjJ8McidJskD/view?usp=drive_link'
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
brands_df_original = pd.read_csv(path)

url = 'https://drive.google.com/file/d/1FYhN_2AzTBFuWcfHaRuKcuCE6CWXsWtG/view?usp=drive_link'
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
orderlines_df_original = pd.read_csv(path)

url = 'https://drive.google.com/file/d/1Vu0q91qZw6lqhIqbjoXYvYAQTmVHh6uZ/view?usp=drive_link'
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
orders_df_original = pd.read_csv(path)

url = 'https://drive.google.com/file/d/1afxwDXfl-7cQ_qLwyDitfcCx3u7WMvkU/view?usp=drive_link'
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
products_df_original = pd.read_csv(path)

# copy of all dataframes
brands_df = brands_df_original.copy()
orderlines_df = orderlines_df_original.copy()
orders_df = orders_df_original.copy()
products_df = products_df_original.copy()

# Initial Exploration
https://colab.research.google.com/drive/1b9XldBi7bbPYyhWoTGkh9IsH23rTjxLP#scrollTo=gOdUXHEQ8_ta

# Data Cleaning

## Orders

**Observation & action plans:**

* Created_date : should be in datetime format
* Total_paid:  
1. Number of missing values : 5
2. 0.00 value for 19334 orders- possible reason?
3. Mean-median gap is high (569.23- 112.99), so extreme high value outliers. Also 75% of orders are for 525.98 or less, while max value is 214,747.53



### Created_date

In [4]:
# converting to datetime format
orders_df["created_date"] = pd.to_datetime(orders_df["created_date"])

In [5]:
# checking for update
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      226909 non-null  int64         
 1   created_date  226909 non-null  datetime64[ns]
 2   total_paid    226904 non-null  float64       
 3   state         226909 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 6.9+ MB


### Total_paid

In [6]:
orders_df['total_paid'].isna().value_counts(normalize=True)
# 5 missing values represents 0.00220% of the rows in our DataFrame

,proportion
total_paid,
False,0.999978
True,0.000022


In [5]:
orders_df.loc[orders_df['total_paid'].isna(),:]

,order_id,created_date,total_paid,state
127701,427314,2017-11-20 18:54:39,NaN,Pending
132013,431655,2017-11-22 12:15:24,NaN,Pending
147316,447411,2017-11-27 10:32:37,NaN,Pending
148833,448966,2017-11-27 18:54:15,NaN,Pending
149434,449596,2017-11-27 21:52:08,NaN,Pending


In [8]:
# to check if they were in stock (4 of them were not in stock)
order_check = (
    orders_df.loc[orders_df["order_id"].isin([427314, 431655, 447411, 448966, 449596])]
    .merge(orderlines_df, left_on="order_id", right_on="id_order")
    .merge(products_df, on="sku")
)

order_check.head()

,order_id,created_date,total_paid,state,id,id_order,product_id,product_quantity,sku,unit_price,date,name,desc,price,promo_price,in_stock,type
0,427314,2017-11-20 18:54:39,NaN,Pending,1416957,427314,0,1,MUJ0023,26.99,2017-11-20 18:40:45,Mujjo Leather Leather iPhone Case 8 Plus / 7 Plus Tan,ultrafine material cover with vegetable tanned leather for iPhone 8 Plus or 7-Plus,44.9,269.891,0,11865403
1,427314,2017-11-20 18:54:39,NaN,Pending,1416966,427314,0,1,APP1696,49.00,2017-11-20 18:44:30,Leather Case Cover Apple iPhone 7 Plus Blue Sea,ultrathin leather case and microfiber premium for iPhone 7 Plus,59,51,0,11865403
2,431655,2017-11-22 12:15:24,NaN,Pending,1421402,431655,0,1,APP1922,154.00,2017-11-22 12:12:16,AirPods Apple Bluetooth Headset for iPhone iPad iPod and Apple Watch,Apple AirPods wireless headsets and cargo transport box,179,1.610.001,1,5384
3,447411,2017-11-27 10:32:37,NaN,Pending,1456182,447411,0,1,LAC0225,180.76,2017-11-27 10:30:28,Lacie Rugged Hard Drive 2TB USB-C Thunderbolt,"2TB hard drive rugged, compact Thunderbolt and USB-C for Mac and PC",209.99,1.699.941,0,11935397
4,448966,2017-11-27 18:54:15,NaN,Pending,1459674,448966,0,1,PAC2266,961.02,2017-11-27 18:03:18,Synology DS918 + NAS Server | 8GB | 8TB (4x2TB) WD Red,NAS server of the Plus Series for companies seeking high performance and the ability to scale memory and,1032.98,9.753.677,0,12175397


Since the missing values are very insignificant (proportion of 0.000022 & status is pending) deleting these rows seems to be a better option

In [6]:
# Deleting rows with missing value
orders_df = orders_df.dropna(axis=0)

In [7]:
orders_df.isna().sum()

,0
order_id,0
created_date,0
total_paid,0
state,0


In [8]:
# to check if all updates done
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 226904 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      226904 non-null  int64         
 1   created_date  226904 non-null  datetime64[ns]
 2   total_paid    226904 non-null  float64       
 3   state         226904 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 8.7+ MB


#### total_paid & state analysis

In [ ]:
# count of total_paid  (Number of orders where 0 total paid is 19334)
orders_df["total_paid"].value_counts().sort_values(ascending= False)

,count
total_paid,
0.00,19334
19.99,2812
29.99,2193
39.99,1504
49.99,1495
...,...
1311.92,1
66.55,1
158.83,1


In [ ]:
# details of total paid in different states (16 completed orders have total _paid as 0)

total_paid_orders = orders_df.loc[(orders_df["total_paid"] == 0.0)]

total_paid_orders.groupby("state")["order_id"].count()

,order_id
state,
Cancelled,35
Completed,16
Pending,6
Place Order,9419
Shopping Basket,9858


In [ ]:
completed_orders = orders_df.loc[(orders_df["state"] == "Completed") & (orders_df["total_paid"] == 0.0), :]
completed_orders

,order_id,created_date,total_paid,state
150,296010,2017-01-09 23:47:00,0.0,Completed
2061,301495,2017-01-03 12:22:02,0.0,Completed
30427,329911,2017-02-27 18:21:01,0.0,Completed
61224,360765,2017-05-30 23:28:57,0.0,Completed
61902,361443,2017-06-01 23:21:54,0.0,Completed
61977,361518,2017-06-02 06:18:27,0.0,Completed
63610,363153,2017-06-07 13:39:13,0.0,Completed
65940,365485,2017-06-14 10:37:02,0.0,Completed
82898,382473,2017-07-26 18:23:09,0.0,Completed
82938,382513,2017-07-26 19:23:21,0.0,Completed


In [ ]:
# to check if the completed order details in orderlines-- (only 2 of the 16 completed orders were present in orderlines)
completed_order_check = (
    orders_df.loc[(orders_df["state"] == "Completed") & (orders_df["total_paid"] == 0.0)]
    .merge(orderlines_df, left_on="order_id", right_on="id_order")
    .merge(products_df,on= "sku")
)

completed_order_check.head()

,order_id,created_date,total_paid,state,id,id_order,product_id,product_quantity,sku,unit_price,date,name,desc,price,promo_price,in_stock,type
0,296010,2017-01-09 23:47:00,0.0,Completed,1138342,296010,0,1,TUC0252,24.99,2017-01-09 23:41:57,"Tucano Nido Hard-Shell Case MacBook Air 13 ""Black",rigid and slicked rubber feet MacBook Air 13 inch casing.,29.9,249.901,0,13835403
1,301495,2017-01-03 12:22:02,0.0,Completed,1123362,301495,0,1,TUC0277,24.99,2017-01-03 12:04:16,"Svolta Tucano MacBook Pro Sleeve bag / Retina / Air 13 ""Black",compact case for MacBook / Air 13 and 13 inches MacBook Retina,29.9,249.901,0,10230


## Orderlines

**Observations & action plan:**
* Date: should be in datetime format
* Product_id: not in use now, only 1 unique value(0), can be deleted.
* Unit_price: should be float
* Product_quantity: max -999 (seems to be outlier since mean is 1.12, also 75% of orders are 1). Can explore more

In [10]:
orderlines_df.head()

,id,id_order,product_id,product_quantity,sku,unit_price,date
0,1119109,299539,0,1,OTT0133,18.99,2017-01-01 00:07:19
1,1119110,299540,0,1,LGE0043,399.00,2017-01-01 00:19:45
2,1119111,299541,0,1,PAR0071,474.05,2017-01-01 00:20:57
3,1119112,299542,0,1,WDT0315,68.39,2017-01-01 00:51:40
4,1119113,299543,0,1,JBL0104,23.74,2017-01-01 01:06:38


In [11]:
# renaming column names
orderlines_df.columns = ["id", "order_id", "product_id","product_quantity", "sku", "unit_price", "orderlines_date"]

### Date (Orderlines_date)

In [12]:
orderlines_df["orderlines_date"] = pd.to_datetime(orderlines_df["orderlines_date"])

### Product_id

In [13]:
# deleting product_id
orderlines_df = orderlines_df.drop("product_id", axis = 1)

### Unit_price

In [14]:
# count of rows with 1 or more decimal points
orderlines_df['unit_price'].str.count(r"\.").value_counts()

,count
unit_price,
1,257814
2,36169


In [15]:
# Count the rows with more than one `.`
mult_decimal_rows = (orderlines_df['unit_price'].str.count("\.")>1).sum()

# Find the percentage of corrupted rows -- 12.30%
percent_corrupted = (100 * mult_decimal_rows / orderlines_df.shape[0])
print(f"{percent_corrupted:.2f}% of the rows in the DataFrame have multiple decimal points for unit_price")

12.30% of the rows in the DataFrame have multiple decimal points for unit_price


<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_3519/2435728751.py:2: SyntaxWarning: invalid escape sequence '\.'
  mult_decimal_rows = (orderlines_df['unit_price'].str.count("\.")>1).sum()


#### Handling unit_price
Since Deleting the price column result in 12.3% of data lost decided to Handle the decimal point by removing first decimal point (from left)
- created a new column unit_price clean


In [16]:
## Handling decimal approach

# creating a new column unit_price_clean to include the corrected price
orderlines_df["unit_price_clean"] = (
    orderlines_df["unit_price"]
    .str.replace(r"\.(?=.*\.)", "", regex=True)
    .astype(float)
)
#  Converting to float & rounding decimal to 2
orderlines_df["unit_price_clean"] = pd.to_numeric(orderlines_df["unit_price_clean"], errors='coerce').round(2)

In [17]:
# checking for updates
orderlines_df[orderlines_df['unit_price'].str.count(r"\.")>1].head()

,id,order_id,product_quantity,sku,unit_price,orderlines_date,unit_price_clean
6,1119115,299544,1,APP1582,1.137.99,2017-01-01 01:17:21,1137.99
11,1119126,299549,1,PAC0929,2.565.99,2017-01-01 02:07:42,2565.99
15,1119131,299553,1,APP1854,3.278.99,2017-01-01 02:14:47,3278.99
43,1119195,299582,1,PAC0961,2.616.99,2017-01-01 08:54:00,2616.99
59,1119214,299596,1,PAC1599,2.873.99,2017-01-01 09:53:11,2873.99


In [18]:
# checking if all updates done
orderlines_df.head()
#orderlines_df.info()

,id,order_id,product_quantity,sku,unit_price,orderlines_date,unit_price_clean
0,1119109,299539,1,OTT0133,18.99,2017-01-01 00:07:19,18.99
1,1119110,299540,1,LGE0043,399.00,2017-01-01 00:19:45,399.00
2,1119111,299541,1,PAR0071,474.05,2017-01-01 00:20:57,474.05
3,1119112,299542,1,WDT0315,68.39,2017-01-01 00:51:40,68.39
4,1119113,299543,1,JBL0104,23.74,2017-01-01 01:06:38,23.74


In [19]:
# deleting unit_price column
orderlines_df = orderlines_df.drop("unit_price", axis = 1)

In [20]:
# renaming unit_price_clean as unit_price
orderlines_df = orderlines_df.rename(columns={"unit_price_clean": "unit_price"})

In [21]:
# checking for updates
orderlines_df.info()
#orderlines_df['unit_price_clean'].str.count(r"\.").value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   id                293983 non-null  int64         
 1   order_id          293983 non-null  int64         
 2   product_quantity  293983 non-null  int64         
 3   sku               293983 non-null  object        
 4   orderlines_date   293983 non-null  datetime64[ns]
 5   unit_price        293983 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(3), object(1)
memory usage: 13.5+ MB


In [22]:
# checking for any outliers or different values

orderlines_df.describe().round(2)

,id,order_id,product_quantity,orderlines_date,unit_price
count,293983.00,293983.00,293983.00,293983,293983.00
mean,1397918.10,419999.12,1.12,2017-09-19 03:19:26.305779712,409.86
min,1119109.00,241319.00,1.00,2017-01-01 00:07:19,-119.00
25%,1262542.50,362258.50,1.00,2017-06-06 16:20:34.500000,35.99
50%,1406940.00,425956.00,1.00,2017-11-13 21:13:53,92.99
75%,1531321.50,478657.00,1.00,2018-01-02 04:47:03,339.58
max,1650203.00,527401.00,999.00,2018-03-14 13:58:36,159989.83
std,153009.56,66344.49,3.40,NaN,837.07


In [24]:
# checking for highest values- only one outlier
orderlines_df.sort_values('unit_price',ascending=False).head(20)

,id,order_id,product_quantity,sku,orderlines_date,unit_price
36979,1197439,331780,1,NEA0009,2017-03-03 22:29:21,159989.83
38317,1200059,332976,6,LAC0223,2017-03-07 15:51:14,15349.00
282442,1632863,520029,1,APP2660,2018-03-02 11:57:52,14725.00
241912,1567694,491691,1,APP2660,2018-01-18 14:40:05,14580.00
232077,1551059,487192,1,APP2660,2018-01-10 00:05:59,14580.00
232052,1551007,487164,1,APP2660,2018-01-09 23:33:16,14580.00
248338,1577659,497375,1,APP2660,2018-01-23 23:55:41,14580.00
255454,1588363,501413,1,APP2660,2018-01-29 18:58:03,14580.00
234834,1555558,488935,4,APP2660,2018-01-11 21:11:34,14580.00
255123,1587927,501275,1,APP2660,2018-01-29 17:18:55,14580.00


In [25]:
unit_price_check = (
    orderlines_df.loc[orderlines_df["unit_price"] > 12500.00]
    .merge(orders_df, on="order_id")
    .merge(products_df, on="sku")
)

unit_price_check.head()

,id,order_id,product_quantity,sku,orderlines_date,unit_price,created_date,total_paid,state,name,desc,price,promo_price,in_stock,type
0,1197439,331780,1,NEA0009,2017-03-03 22:29:21,159989.83,2017-03-03 22:29:21,159989.83,Shopping Basket,Netatmo home thermostat for iPhone and iPad,Thermostat iPhone iPad wifi Mac design Stark.,179,174.989,0,11905404
1,1200059,332976,6,LAC0223,2017-03-07 15:51:14,15349.00,2017-03-07 15:51:14,92094.00,Shopping Basket,12big LaCie Hard Disk 120TB RAID Thunderbolt 3 USB-C,12 120TB hard drive bays with RAID 5 two ports Thunderbolt 3 (40Gb / s) and 5 years warranty for Mac and PC,9799,78.999.944,0,11935397
2,1241776,352790,3,LAC0223,2017-05-04 18:59:32,14099.00,2017-05-04 18:59:32,48684.97,Shopping Basket,12big LaCie Hard Disk 120TB RAID Thunderbolt 3 USB-C,12 120TB hard drive bays with RAID 5 two ports Thunderbolt 3 (40Gb / s) and 5 years warranty for Mac and PC,9799,78.999.944,0,11935397
3,1244301,354084,4,LAC0223,2017-05-09 09:04:18,14099.00,2017-05-09 09:04:18,56396.00,Shopping Basket,12big LaCie Hard Disk 120TB RAID Thunderbolt 3 USB-C,12 120TB hard drive bays with RAID 5 two ports Thunderbolt 3 (40Gb / s) and 5 years warranty for Mac and PC,9799,78.999.944,0,11935397
4,1551007,487164,1,APP2660,2018-01-09 23:33:16,14580.00,2018-01-09 23:33:16,14580.00,Shopping Basket,"Apple iMac Pro 27 ""18-core Intel Xeon W 23GHz | 128GB | 4TB SSD | Radeon Pro Vega 64",Pro iMac 27 inch screen Retina 5K and Intel Xeon processor W of 23GHz,15339,144.190.049,0,118692158


In [26]:
# checking for lowest values- (0.0 for 569 orders & -119.0 for 1)
orderlines_df.sort_values('unit_price',ascending=False).tail(20)

,id,order_id,product_quantity,sku,orderlines_date,unit_price
58512,1236190,350048,1,LIBRO,2017-04-26 16:44:52,0.0
58515,1236193,350050,4,LIBRO,2017-04-26 16:47:47,0.0
58523,1236205,350058,1,LIBRO,2017-04-26 16:56:08,0.0
58485,1236137,350024,1,LIBRO,2017-04-26 15:54:14,0.0
58641,1236407,350139,1,LIBRO,2017-04-26 20:53:35,0.0
221812,1533952,480296,1,ZAG0022,2018-01-02 21:07:41,0.0
58529,1236215,350062,1,LIBRO,2017-04-26 17:06:32,0.0
58487,1236140,350025,2,LIBRO,2017-04-26 15:55:00,0.0
195581,1486724,458985,1,QNA0202,2017-12-10 16:01:43,0.0
195582,1486725,458985,1,QNA0146,2017-12-10 16:01:43,0.0


In [27]:
unit_price_check = (
    orderlines_df.loc[orderlines_df["unit_price"] <= 0]
    .merge(orders_df, on="order_id")
    .merge(products_df, on="sku")
)

unit_price_check.head()

,id,order_id,product_quantity,sku,orderlines_date,unit_price,created_date,total_paid,state,name,desc,price,promo_price,in_stock,type
0,1227566,345934,1,KIN0153-2,2017-04-13 13:47:21,0.0,2017-04-13 13:54:40,88.98,Cancelled,Mac memory Kingston 16GB (2x8GB) SO-DIMM DDR3åÊ1600Mhz,RAM 16GB MacBook Pro iMac (2013) and Mac mini (2012).,149.98,1.469.896,0,1364
1,1227590,345957,1,WDT0347,2017-04-13 14:44:05,0.0,2017-04-13 15:00:18,3.99,Cancelled,WD Blue 250GB SATA 3 SSD Disk 7mm,SSD 250GB SATA Hard Disk 3.0 (6Gb / s) for Mac and PC,99,859.935,0,12215397
2,1268645,365886,1,APP1465,2017-06-15 12:48:54,-119.0,2017-06-15 12:51:18,30.00,Cancelled,Spanish Keyboard Keyboard Magic Apple Mac (OEM),Spanish Keyboard Mac and Apple iPad Ultrathin Bluetooth (unboxed),119,999.944,1,13855401
3,1299125,380508,1,APP2264,2017-07-21 17:29:57,0.0,2017-07-21 17:31:46,0.00,Cancelled,"Apple Macbook Pro 13 ""Core i7 Touch Bar 35GHz | 16GB | 512GB SSD Space Gray",New MacBook Pro 13 inch Touch Bar 35 GHz Core i7 with 16GB of RAM and 512GB of SSD PCIe,2849,27.070.047,1,"1,02E+12"
4,1299125,380508,1,APP2264,2017-07-21 17:29:57,0.0,2017-07-21 17:31:46,0.00,Cancelled,"Apple Macbook Pro 13 ""Core i7 Touch Bar 35GHz | 16GB | 512GB SSD Space Gray",New MacBook Pro 13 inch Touch Bar 35 GHz Core i7 with 16GB of RAM and 512GB of SSD PCIe,2849,27.070.047,1,"1,02E+12"


In [28]:
unit_price_check = (
    orderlines_df.loc[orderlines_df["unit_price"] <= 0]
    .merge(orders_df, on="order_id")
    .merge(products_df, on="sku")
)[["sku","order_id","unit_price","price","state","total_paid","in_stock","name","desc"]]

unit_price_check.head()

,sku,order_id,unit_price,price,state,total_paid,in_stock,name,desc
0,KIN0153-2,345934,0.0,149.98,Cancelled,88.98,0,Mac memory Kingston 16GB (2x8GB) SO-DIMM DDR3åÊ1600Mhz,RAM 16GB MacBook Pro iMac (2013) and Mac mini (2012).
1,WDT0347,345957,0.0,99,Cancelled,3.99,0,WD Blue 250GB SATA 3 SSD Disk 7mm,SSD 250GB SATA Hard Disk 3.0 (6Gb / s) for Mac and PC
2,APP1465,365886,-119.0,119,Cancelled,30.00,1,Spanish Keyboard Keyboard Magic Apple Mac (OEM),Spanish Keyboard Mac and Apple iPad Ultrathin Bluetooth (unboxed)
3,APP2264,380508,0.0,2849,Cancelled,0.00,1,"Apple Macbook Pro 13 ""Core i7 Touch Bar 35GHz | 16GB | 512GB SSD Space Gray",New MacBook Pro 13 inch Touch Bar 35 GHz Core i7 with 16GB of RAM and 512GB of SSD PCIe
4,APP2264,380508,0.0,2849,Cancelled,0.00,1,"Apple Macbook Pro 13 ""Core i7 Touch Bar 35GHz | 16GB | 512GB SSD Space Gray",New MacBook Pro 13 inch Touch Bar 35 GHz Core i7 with 16GB of RAM and 512GB of SSD PCIe


Further questions:
* should we remove the outlier for unit price:
1. highest value: 159989.83?
2. lowest value: -119.00


### Order_id & sku combination analysis
  - these combinations are not unique

In [42]:
# check if all order_id and sku combinations are unique
orderlines_df.groupby(["order_id", "sku"]).size()

,,0
order_id,sku,
241319,JBL0123,1
241355,WAC0171,1
241423,LAC0212,1
242832,PAR0074,1
243330,OWC0074,1
...,...,...
527397,JBL0122,1
527398,JBL0122,1
527399,PAC0653,1


In [43]:
# step 1: finding rows where the combo has appeared before using .duplicated()

is_duplicate = orderlines_df.duplicated(subset=['order_id', 'sku']).any()

print(f"Are there duplicates in order_id + sku? {is_duplicate}")

Are there duplicates in order_id + sku? True


In [44]:
# step 2: since they are not unique combination, checking on occurences of duplicate

#'keep=False' shows all occurrences of the duplicate, not just the second one
duplicates = orderlines_df[orderlines_df.duplicated(subset=['order_id', 'sku'], keep=False)]

# Sorting by order_id to see them side-by-side
duplicates.sort_values("order_id").head(20)

,id,order_id,product_quantity,sku,orderlines_date,unit_price_clean
103477,1316206,377561,1,APP1919,2017-08-11 09:53:55,55.99
110159,1328322,377561,1,APP1919,2017-08-30 10:27:13,37.99
115139,1349473,398536,1,ALL0010,2017-09-10 23:29:16,15.99
117270,1353361,398536,1,ALL0010,2017-09-15 17:15:15,15.99
181441,1467969,451840,1,SXA0011,2017-11-29 14:27:08,34.99
181446,1467978,451840,1,SXA0011,2017-11-29 14:35:35,42.34
188412,1477473,455540,1,APP1919,2017-12-05 09:51:09,50.39
188414,1477475,455540,1,APP1919,2017-12-05 09:51:20,50.39
188901,1478179,455794,1,APP1919,2017-12-05 16:18:59,50.39
188902,1478180,455794,1,APP1919,2017-12-05 16:19:15,50.39


## Products

**Observations & Action plan:**
* Number of duplicated rows: 8746
* Desc: Number of missing values : 7 (can possibly impute with name)
* Price:
1. should be float
2. Number of missing values : 46
* Promo_price: should be float
* Type:
1. Number of missing values : 50
2. Type should be numerical as per the Data info
* In_stock:  values of 0 & 1 to indicate if present in stock or not , can turn to str values if needed for better visualisation.


### Duplicates

In [32]:
# deleting duplicates
products_df = products_df.drop_duplicates()

In [33]:
# checking for update
products_df.duplicated().sum()

np.int64(0)

In [34]:
# since sku is unique value, checking for duplicates in this column
products_df[products_df["sku"].duplicated(keep=False)]

,sku,name,desc,price,promo_price,in_stock,type
7992,APP1197,"Apple iMac 21.5 ""Core i5 31 GHz Retina display 4K | 8GB | 1TB",Desktop Apple iMac 21.5 inch i5 31 GHz Retina display 4K RAM 8GB 1TB (MK452Y / A).,1729,1305.59,0,1282
8000,APP1197,"Apple iMac 21.5 ""Core i5 31 GHz Retina display 4K | 8GB | 1TB",Desktop Apple iMac 21.5 inch i5 31 GHz Retina display 4K RAM 8GB 1TB (MK452Y / A).,NaN,1305.59,0,1282


In [35]:
# Filter out rows where the SKU is duplicated AND the price is null
products_df = products_df.loc[~(products_df["sku"].duplicated(keep=False) & products_df["price"].isna())]

In [36]:
products_df = products_df.reset_index(drop=True) # reseting the index

### Desc

In [37]:
products_df['desc'].isna().value_counts(normalize=True)

,proportion
desc,
False,0.999338
True,0.000662


In [38]:
products_df.loc[products_df['desc'].isna(), :]

,sku,name,desc,price,promo_price,in_stock,type
7675,WDT0211-A,"Open - Purple 2TB WD 35 ""PC Security Mac hard drive and NAS",NaN,107,814.659,0,1298
7677,APP1622-A,"Open - Apple Smart Keyboard Pro Keyboard Folio iPad 9.7 """,NaN,1.568.206,1.568.206,0,1298
9099,PAC2334,Synology DS718 + NAS Server | 10GB RAM,NaN,566.35,5.659.896,0,12175397
9408,KAN0034-A,"Open - Kanex USB-C Gigabit Ethernet Adapter MacBook 12 """,NaN,29.99,237.925,0,1298
9744,HTE0025,Hyper Pearl 1600mAh battery Mini USB Mirror and Comic Blond,NaN,24.99,22.99,1,1515
9865,OTT0200,OtterBox External Battery Power Pack 20000 mAHr,NaN,79.99,56.99,1,1515
9943,HOW0001-A,Open - Honeywell thermostat Lyric zonificador T6 Intelligent Wireless (cable),NaN,199.99,1.441.174,0,11905404


Since there are only 7 missing values & the 'name' & 'desc' quite similiar in the contents, instead of deleting missing values, replacing it with corresponding info from the 'name'

In [39]:
products_df.loc[products_df['desc'].isna(), 'desc'] = products_df.loc[products_df['desc'].isna(), 'name']

In [40]:
# checking for update
products_df['desc'].isna().value_counts()

,count
desc,
False,10579


### Price

In [41]:
## Missing values
products_df['price'].isna().value_counts()

,count
price,
False,10534
True,45


In [42]:
products_df['price'].isna().value_counts(normalize=True)

,proportion
price,
False,0.995746
True,0.004254


In [43]:
print(f"The missing values in price are {(products_df.price.isna().value_counts(normalize=True).iloc[1] * 100).round(2)}% of all rows in the DataFrame")

The missing values in price are 0.43% of all rows in the DataFrame


#### Handling price
* Missing data: Deleting 45 rows (0.43% of data)
* Multiple decimal points?
1. Deleting rows with multiple decimal values (5.12 % of data) or
2. Handling the multiple decimal values

Millesimal Precision: Some databases store currency with 3 decimal places (common in accounting software). Converting 4694.994 to a float keeps that precision, which you can later round to 2 decimals using .round(2).

In [44]:
# Deleting columns with missisng values
products_df = products_df.dropna(subset=['price'])

In [ ]:
## Data type_ handling multiple decimal values

In [45]:
products_df['price'].str.count(r"\.").value_counts()

,count
price,
1,6942
0,3215
2,377


In [46]:
products_df.loc[(products_df.price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.price.str.contains(r"\d+\.\d{3,}")), :].shape#[0]

(542, 7)

In [47]:
price_problems_number = products_df.loc[(products_df.price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.price.str.contains(r"\d+\.\d{3,}")), :].shape[0]
print(f"The column price has in total {price_problems_number} incorrect values.\nThis is {round(((price_problems_number / products_df.shape[0]) * 100), 2)}% of the rows of the DataFrame")

The column price has in total 542 incorrect values.
This is 5.15% of the rows of the DataFrame


In [48]:
# updating multiple decimal points & converting to float
mask = products_df["price"].astype(str).str.contains(r"\.\d\d\d\.\d\d\d$", na=False)

products_df["price"] = products_df["price"].astype("object")

products_df.loc[mask, "price"] = (
    products_df.loc[mask, "price"]
    .astype(str)
    .str.replace(".", "", regex = False)
    .astype(float)
    / 100000
)

In [49]:
#  Converting to float & rounding decimal to 2
products_df["price"] = pd.to_numeric(products_df["price"], errors='coerce').round(2)

In [50]:
# checking for updates
products_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10534 entries, 0 to 10578
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sku          10534 non-null  object 
 1   name         10534 non-null  object 
 2   desc         10534 non-null  object 
 3   price        10534 non-null  float64
 4   promo_price  10534 non-null  object 
 5   in_stock     10534 non-null  int64  
 6   type         10484 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 658.4+ KB


In [ ]:
# quality check of updated data
products_df.describe().round(2)

In [ ]:
products_df.sort_values("price",ascending= False).head(20)

### Promo_price

#### Handling promo_price
* Deleting the rows (92.01% of data lost )
* Deleting the column 'promo_price' can be a better approach as it doesnot provide further insights  for analysis.



In [51]:
promo_problems_number = products_df.loc[(products_df.promo_price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.promo_price.str.contains(r"\d+\.\d{3,}")), :].shape[0]
print(f"The column promo_price has in total {promo_problems_number} wrong values.\nThis is {round(((promo_problems_number / products_df.shape[0]) * 100), 2)}% of the rows of the DataFrame")

The column promo_price has in total 9691 wrong values.
This is 92.0% of the rows of the DataFrame


In [52]:
# Deleting column 'promo_price'
products_df = products_df.drop(columns=["promo_price"])

In [53]:
# checking for updates
products_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10534 entries, 0 to 10578
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   sku       10534 non-null  object 
 1   name      10534 non-null  object 
 2   desc      10534 non-null  object 
 3   price     10534 non-null  float64
 4   in_stock  10534 non-null  int64  
 5   type      10484 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 576.1+ KB


### Type

Type isn’t an essential piece of data for the analysis & is therefore allowed to carry missing values. Might be an optional route to category creation, but that can still be done using name and desc to categorize those rows.

In [54]:
# Missing values (50) _ no action
# Data type can be explored (since it contains "E", read as object time, so can check for patterns)

In [55]:
products_df["type"] = products_df["type"].str.replace(",", ".")

In [56]:
products_df["type"] = pd.to_numeric(products_df["type"])

In [57]:
# checking for updates
products_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10534 entries, 0 to 10578
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   sku       10534 non-null  object 
 1   name      10534 non-null  object 
 2   desc      10534 non-null  object 
 3   price     10534 non-null  float64
 4   in_stock  10534 non-null  int64  
 5   type      10484 non-null  float64
dtypes: float64(2), int64(1), object(3)
memory usage: 576.1+ KB


In [ ]:
products_df.describe().round(2)

# Exporting cleaned data

In [30]:
from google.colab import files

In [ ]:
orders_df.to_csv("orders_cl.csv", index=False)
files.download("orders_cl.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [58]:
#orders_df.to_csv("orders_cl.csv", index=False)
#files.download("orders_cl.csv")

#orderlines_df.to_csv("orderlines_cl.csv", index=False)
#files.download("orderlines_cl.csv")

products_df.to_csv("products_cl.csv", index=False)
files.download("products_cl.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>